In [29]:
import os
import pandas as pd
from IPython.display import display
from tqdm import tqdm
import re

folder = os.getcwd()
excels = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".xlsx")]

In [31]:
len(excels)

47

In [32]:
import pandas as pd
from tqdm import tqdm

# 1. Словарь для переименования
rename_map = {
    'Material No.': 'Код материалов',
    '物料编码': 'Код материалов',
    'Qty': 'Кол-во',
    '件数': 'Кол-во',
    'Units': 'Кол-во',
    'Item and Spec': 'Наименование и спецификация',
    '名称及规格': 'Наименование и спецификация',
    'Section': 'Section',
    'Models': 'Models'
}

# 2. Список того, что нужно ВЫКИНУТЬ (независимо от переименования)
exclude_cols = {'Примечание', 'Remark', '备注', 'remark', 'Rev.'}

# Список нужных нам итоговых названий (из значений rename_map)
target_cols = set(rename_map.values())

dfs = []

for pth in tqdm(excels):
    # Читаем файл
    df_i = pd.read_excel(pth)
    
    # Сначала переименовываем колонки по словарю
    df_i = df_i.rename(columns=rename_map)
    
    # Фильтруем колонки: оставляем только те, что:
    # 1. Есть в нашем целевом списке (target_cols)
    # 2. И при этом НЕ входят в список исключений (exclude_cols)
    cols_to_keep = [c for c in df_i.columns if c in target_cols and c not in exclude_cols]
    
    # Добавляем в список только нужные данные
    dfs.append(df_i[cols_to_keep])

# Склеиваем
df_all = pd.concat(dfs, ignore_index=True)

print("Итоговые колонки:", df_all.columns.tolist())

100%|██████████| 47/47 [00:02<00:00, 18.72it/s]

Итоговые колонки: ['Код материалов', 'Кол-во', 'Наименование и спецификация', 'Section', 'Models']


In [33]:
df_all

,Код материалов,Кол-во,Наименование и спецификация,Section,Models
0,1000000867,1.0,Дизельный двигатель,Энергетический система ZS1623RT,ZS1623RT
1,1000100391,1.0,Фильтр воздуха в сборе,Энергетический система ZS1623RT,ZS1623RT
2,1000200483,1.0,Радиатор в сборе,Энергетический система ZS1623RT,ZS1623RT
3,1000300735,1.0,Расширительный бак,Энергетический система ZS1623RT,ZS1623RT
4,1009900968,1.0,Пружинная муфта в сборе,Энергетический система ZS1623RT,ZS1623RT
...,...,...,...,...,...
33559,1140305984,1.0,Шланг в сборе,Трубопровод гидравлической системы в сборе (тр...,ZX23AE
33560,1140305985,1.0,Шланг в сборе,Трубопровод гидравлической системы в сборе (тр...,ZX23AE
33561,1140305986,2.0,Шланг в сборе,Трубопровод гидравлической системы в сборе (тр...,ZX23AE
33562,1149900260,1.0,Переходное соединение,Сборка главного клапана платформы (сборка аксе...,ZX23AE


In [34]:
def join_ordered(series: pd.Series):
    all_parts = []

    for item in series.dropna(): 
        parts = str(item).split(';') 
        for p in parts:
            p_clean = re.sub(r'\s+', '', p).upper()
            if p_clean:
                all_parts.append(p_clean)

    seen = set()
    result = []
    for v in all_parts:
        if v not in seen:
            seen.add(v)
            result.append(v)

    return "; ".join(result) if result else None


def join_ordered_description(series: pd.Series):
    all_parts = []

    for item in series.dropna():
        text = str(item).strip()
        if not text:
            continue

        parts = [p.strip() for p in text.split(";")]

        for p in parts:
            p_clean = re.sub(r"\s+", " ", p).strip()
            if p_clean:
                all_parts.append(p_clean)

    seen = set()
    result = []

    for v in all_parts:
        key = re.sub(r"\s+", " ", v).strip().lower()

        if key not in seen:
            seen.add(key)
            result.append(v)

    return "; ".join(result) if result else None


In [35]:
def merge_parts(df):
    df = df.copy()

    agg_map = {
        'Кол-во': 'sum',
        "Наименование и спецификация": join_ordered_description,
        "Section": join_ordered_description,
        "Models": join_ordered
    }

    df_agg = df.groupby("Код материалов", as_index=False).agg(agg_map)

    return df_agg

In [36]:
df_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 33564 entries, 0 to 33563
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Код материалов               33564 non-null  object 
 1   Кол-во                       33564 non-null  float64
 2   Наименование и спецификация  33555 non-null  str    
 3   Section                      33535 non-null  str    
 4   Models                       33564 non-null  str    
dtypes: float64(1), object(1), str(3)
memory usage: 1.3+ MB


In [37]:
df_merged = merge_parts(df_all) 

In [38]:
df_merged

,Код материалов,Кол-во,Наименование и спецификация,Section,Models
0,204060296,9.0,Great wall vehicle gear oil; Автомобильное тра...,Chassis 00773202800000000; Шасси 0077370280000...,ZA10RJE; ZA20JE
1,1000000867,1.0,Дизельный двигатель,Энергетический система ZS1623RT,ZS1623RT
2,1000100391,1.0,Фильтр воздуха в сборе,Энергетический система ZS1623RT,ZS1623RT
3,1000200483,1.0,Радиатор в сборе,Энергетический система ZS1623RT,ZS1623RT
4,1000300735,1.0,Расширительный бак,Энергетический система ZS1623RT,ZS1623RT
...,...,...,...,...,...
13528,NO-NUMBER(5),1.0,壳体,回转机构(回转马达),ZT26JS-V
13529,NO-NUMBER(6),1.0,推力轴承,回转机构(回转马达),ZT26JS-V
13530,NO-NUMBER(7),1.0,联动轴,回转机构(回转马达),ZT26JS-V
13531,NO-NUMBER(8),1.0,隔盘,回转机构(回转马达),ZT26JS-V


In [39]:
df_merged.to_excel(r"Zoomlion_модели.xlsx", sheet_name="Parts", index=False)

C:\Users\a.vorona\AppData\Local\Temp\ipykernel_13680\2114824825.py:1: UserWarning: Cell contents too long (33509), truncated to 32767 characters
  df_merged.to_excel(r"Zoomlion_модели.xlsx", sheet_name="Parts", index=False)
